# Week 11: Training Data Collection for Learned Query Pooler

This notebook collects **100+ training samples** for the Learned Query Pooler by:
1. Generating diverse benchmark questions across multiple domains
2. Simulating CCE spikes with realistic confused token distributions
3. Creating proper train/test splits

## Why More Data?
- Week 10: 17 samples → Overfitting (tested on training data)
- Week 11 v1: 21 train samples → Learned method performed worse than heuristic
- **This notebook: 150+ samples → Proper deep learning scale**

In [ ]:
import json
import random
import numpy as np
from dataclasses import dataclass, asdict
from typing import List, Dict
from itertools import product

# Set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Data collection notebook initialized")

## 1. Define File Repository Structure

Expanded file list covering multiple frameworks and domains.

In [ ]:
# Expanded file list with clear domain groupings
FILE_REGISTRY = {
    # Authentication & Security
    "auth": [
        "auth_service.py",
        "auth/token_manager.py",
        "auth/jwt_handler.py",
        "auth/oauth_provider.py",
        "auth/session_store.py",
        "auth/permissions.py",
        "security/encryption.py",
        "security/hashing.py",
    ],
    
    # API & Routing
    "api": [
        "api_router.py",
        "api/endpoints.py",
        "api/middleware.py",
        "api/validators.py",
        "api/serializers.py",
        "fastapi/routing.py",
        "fastapi/params.py",
        "fastapi/dependencies/utils.py",
        "fastapi/background.py",
    ],
    
    # Database
    "database": [
        "database_service.py",
        "db/connection.py",
        "db/models.py",
        "db/queries.py",
        "db/migrations.py",
        "db/transactions.py",
    ],
    
    # Caching
    "cache": [
        "cache_service.py",
        "cache/redis_client.py",
        "cache/memory_cache.py",
        "cache/decorators.py",
    ],
    
    # Events & Messaging
    "events": [
        "event_system.py",
        "events/dispatcher.py",
        "events/handlers.py",
        "events/queue.py",
        "messaging/publisher.py",
        "messaging/subscriber.py",
    ],
    
    # HTTP Client (Requests)
    "http": [
        "requests/sessions.py",
        "requests/api.py",
        "requests/models.py",
        "requests/adapters.py",
        "requests/auth.py",
        "requests/cookies.py",
        "http/client.py",
        "http/response.py",
    ],
    
    # Flask Framework
    "flask": [
        "src/flask/app.py",
        "src/flask/blueprints.py",
        "src/flask/config.py",
        "src/flask/ctx.py",
        "src/flask/globals.py",
        "src/flask/helpers.py",
        "src/flask/scaffold.py",
        "src/flask/views.py",
    ],
    
    # Utilities
    "utils": [
        "utils/helpers.py",
        "utils/validators.py",
        "utils/formatters.py",
        "utils/logger.py",
        "config/settings.py",
        "config/constants.py",
    ],
}

# Flatten to single list
ALL_FILES = []
for domain_files in FILE_REGISTRY.values():
    ALL_FILES.extend(domain_files)

print(f"Total files: {len(ALL_FILES)}")
for domain, files in FILE_REGISTRY.items():
    print(f"  {domain}: {len(files)} files")

## 2. Define Question Templates

Templates for generating diverse questions with associated keywords and target files.

In [ ]:
@dataclass
class QuestionTemplate:
    """Template for generating training samples."""
    domain: str
    question_patterns: List[str]  # Question templates with {placeholders}
    context_patterns: List[str]   # Context templates
    keywords: List[List[str]]     # Groups of related keywords (confused tokens)
    target_files: List[str]       # Files that should be retrieved
    difficulty: str = "medium"


# Comprehensive question templates by domain
QUESTION_TEMPLATES = [
    # =========== AUTHENTICATION ===========
    QuestionTemplate(
        domain="auth",
        question_patterns=[
            "How does {concept} work in the authentication system?",
            "Where is {concept} implemented?",
            "How do I use {concept} for user authentication?",
            "What handles {concept} in the codebase?",
        ],
        context_patterns=[
            "The authentication system uses {concept} to",
            "When authenticating users, {concept} is handled by",
            "For {concept}, the system",
        ],
        keywords=[
            ["token", "generate", "jwt", "sign", "payload"],
            ["verify", "validate", "decode", "signature", "check"],
            ["session", "store", "cookie", "expire", "refresh"],
            ["password", "hash", "bcrypt", "salt", "compare"],
            ["login", "authenticate", "credentials", "user", "password"],
            ["logout", "invalidate", "revoke", "clear", "session"],
            ["permission", "role", "access", "authorize", "check"],
            ["oauth", "provider", "callback", "token", "scope"],
        ],
        target_files=["auth_service.py", "auth/token_manager.py", "auth/jwt_handler.py"],
        difficulty="easy",
    ),
    QuestionTemplate(
        domain="auth",
        question_patterns=[
            "How is OAuth {action} implemented?",
            "Where does OAuth {action} happen?",
        ],
        context_patterns=[
            "OAuth {action} is performed by",
            "For OAuth {action}, the system uses",
        ],
        keywords=[
            ["oauth", "authorize", "redirect", "callback", "code"],
            ["oauth", "token", "exchange", "refresh", "access"],
            ["oauth", "scope", "permission", "grant", "consent"],
        ],
        target_files=["auth/oauth_provider.py", "auth_service.py"],
        difficulty="hard",
    ),
    QuestionTemplate(
        domain="auth",
        question_patterns=[
            "How does session {action} work?",
            "Where are sessions {action}?",
        ],
        context_patterns=[
            "Sessions are {action} using",
            "For session {action}, we use",
        ],
        keywords=[
            ["session", "create", "store", "id", "data"],
            ["session", "retrieve", "get", "load", "fetch"],
            ["session", "delete", "destroy", "expire", "invalidate"],
        ],
        target_files=["auth/session_store.py", "requests/sessions.py"],
        difficulty="medium",
    ),
    
    # =========== API & ROUTING ===========
    QuestionTemplate(
        domain="api",
        question_patterns=[
            "How are API {concept} defined?",
            "Where do I configure {concept}?",
            "How does {concept} work in the API layer?",
        ],
        context_patterns=[
            "API {concept} are defined in",
            "The {concept} configuration uses",
        ],
        keywords=[
            ["route", "endpoint", "path", "url", "method"],
            ["router", "register", "prefix", "include", "mount"],
            ["handler", "controller", "view", "action", "dispatch"],
            ["middleware", "intercept", "before", "after", "filter"],
            ["validator", "schema", "pydantic", "check", "validate"],
        ],
        target_files=["api_router.py", "api/endpoints.py", "fastapi/routing.py"],
        difficulty="easy",
    ),
    QuestionTemplate(
        domain="api",
        question_patterns=[
            "How does dependency injection work with {concept}?",
            "Where is {concept} dependency defined?",
        ],
        context_patterns=[
            "Dependency injection for {concept} uses",
            "The {concept} dependency is injected via",
        ],
        keywords=[
            ["depends", "inject", "dependency", "provide", "resolve"],
            ["param", "query", "body", "header", "path"],
            ["factory", "singleton", "scope", "lifecycle", "create"],
        ],
        target_files=["fastapi/dependencies/utils.py", "fastapi/params.py"],
        difficulty="medium",
    ),
    QuestionTemplate(
        domain="api",
        question_patterns=[
            "How do background tasks handle {concept}?",
            "Where is {concept} processed asynchronously?",
        ],
        context_patterns=[
            "Background processing of {concept} uses",
            "Async {concept} is handled by",
        ],
        keywords=[
            ["background", "task", "async", "queue", "worker"],
            ["schedule", "delay", "defer", "later", "cron"],
            ["celery", "job", "execute", "run", "process"],
        ],
        target_files=["fastapi/background.py", "events/queue.py"],
        difficulty="medium",
    ),
    
    # =========== DATABASE ===========
    QuestionTemplate(
        domain="database",
        question_patterns=[
            "How does the database handle {concept}?",
            "Where is {concept} implemented in the DB layer?",
            "How do I use {concept} with the database?",
        ],
        context_patterns=[
            "Database {concept} is managed by",
            "For {concept}, the database layer uses",
        ],
        keywords=[
            ["connection", "pool", "connect", "close", "acquire"],
            ["query", "execute", "select", "fetch", "cursor"],
            ["transaction", "commit", "rollback", "atomic", "savepoint"],
            ["model", "table", "column", "field", "schema"],
            ["migrate", "migration", "alter", "upgrade", "version"],
            ["insert", "update", "delete", "upsert", "bulk"],
        ],
        target_files=["database_service.py", "db/connection.py", "db/queries.py"],
        difficulty="medium",
    ),
    QuestionTemplate(
        domain="database",
        question_patterns=[
            "How are database models for {entity} defined?",
            "Where is the {entity} model?",
        ],
        context_patterns=[
            "The {entity} model defines",
            "Database model for {entity} includes",
        ],
        keywords=[
            ["model", "class", "table", "column", "relationship"],
            ["field", "type", "integer", "string", "foreign"],
            ["primary", "key", "index", "unique", "constraint"],
        ],
        target_files=["db/models.py", "database_service.py"],
        difficulty="easy",
    ),
    
    # =========== CACHING ===========
    QuestionTemplate(
        domain="cache",
        question_patterns=[
            "How does caching work for {concept}?",
            "Where is {concept} cached?",
            "How do I cache {concept}?",
        ],
        context_patterns=[
            "Caching {concept} uses",
            "The cache for {concept} is managed by",
        ],
        keywords=[
            ["cache", "get", "set", "key", "value"],
            ["expire", "ttl", "timeout", "invalidate", "refresh"],
            ["redis", "client", "connect", "store", "retrieve"],
            ["memoize", "decorator", "wrap", "cached", "remember"],
            ["delete", "clear", "purge", "flush", "remove"],
        ],
        target_files=["cache_service.py", "cache/redis_client.py", "cache/decorators.py"],
        difficulty="easy",
    ),
    
    # =========== EVENTS ===========
    QuestionTemplate(
        domain="events",
        question_patterns=[
            "How does the event system handle {concept}?",
            "Where are {concept} events processed?",
            "How do I implement {concept} with events?",
        ],
        context_patterns=[
            "Event handling for {concept} uses",
            "The {concept} event is processed by",
        ],
        keywords=[
            ["event", "emit", "fire", "trigger", "dispatch"],
            ["listener", "handler", "callback", "subscribe", "on"],
            ["publish", "broadcast", "notify", "send", "channel"],
            ["queue", "async", "defer", "process", "worker"],
            ["register", "bind", "attach", "hook", "connect"],
        ],
        target_files=["event_system.py", "events/dispatcher.py", "events/handlers.py"],
        difficulty="medium",
    ),
    
    # =========== HTTP CLIENT ===========
    QuestionTemplate(
        domain="http",
        question_patterns=[
            "How does the HTTP client handle {concept}?",
            "Where is {concept} configured for requests?",
            "How do I make {concept} HTTP requests?",
        ],
        context_patterns=[
            "HTTP {concept} is handled by",
            "For {concept} requests, we use",
        ],
        keywords=[
            ["request", "get", "post", "put", "delete"],
            ["session", "persistent", "connection", "keep-alive", "pool"],
            ["response", "status", "body", "headers", "content"],
            ["timeout", "retry", "backoff", "error", "exception"],
            ["auth", "header", "bearer", "basic", "credentials"],
            ["cookie", "jar", "store", "send", "receive"],
            ["adapter", "mount", "transport", "protocol", "https"],
        ],
        target_files=["requests/sessions.py", "requests/api.py", "requests/models.py"],
        difficulty="medium",
    ),
    
    # =========== FLASK ===========
    QuestionTemplate(
        domain="flask",
        question_patterns=[
            "How does Flask handle {concept}?",
            "Where is {concept} configured in Flask?",
            "How do I use {concept} in Flask?",
        ],
        context_patterns=[
            "Flask {concept} is managed by",
            "For {concept}, Flask uses",
        ],
        keywords=[
            ["app", "create", "factory", "instance", "wsgi"],
            ["blueprint", "register", "prefix", "url", "module"],
            ["config", "setting", "environment", "debug", "secret"],
            ["context", "request", "application", "push", "pop"],
            ["global", "g", "current_app", "request", "session"],
            ["view", "route", "decorator", "endpoint", "method"],
        ],
        target_files=["src/flask/app.py", "src/flask/blueprints.py", "src/flask/ctx.py"],
        difficulty="medium",
    ),
    QuestionTemplate(
        domain="flask",
        question_patterns=[
            "How does Flask's request context work with {concept}?",
            "Where is {concept} stored in Flask's context?",
        ],
        context_patterns=[
            "Flask context for {concept} uses",
            "Request context stores {concept} in",
        ],
        keywords=[
            ["context", "local", "stack", "push", "pop"],
            ["request", "current", "proxy", "global", "thread"],
            ["application", "app", "context", "manager", "enter"],
        ],
        target_files=["src/flask/ctx.py", "src/flask/globals.py"],
        difficulty="hard",
    ),
]

print(f"Defined {len(QUESTION_TEMPLATES)} question templates")

## 3. Generate Training Samples

In [ ]:
@dataclass
class TrainingSample:
    """A training sample for the Learned Query Pooler."""
    sample_id: str
    confused_tokens: List[str]
    confused_probs: List[float]
    original_query: str
    generated_context: str
    relevant_files: List[str]
    domain: str
    difficulty: str


def generate_probs(n: int) -> List[float]:
    """Generate realistic probability distribution for confused tokens."""
    # Zipf-like distribution (first token most probable)
    raw = [1.0 / (i + 1) ** 0.8 for i in range(n)]
    # Add noise
    raw = [r + random.uniform(-0.05, 0.05) for r in raw]
    raw = [max(0.05, r) for r in raw]
    # Normalize
    total = sum(raw)
    return [round(r / total, 3) for r in raw]


def generate_samples_from_template(template: QuestionTemplate, num_samples: int = 10) -> List[TrainingSample]:
    """Generate multiple samples from a single template."""
    samples = []
    sample_idx = 0
    
    for keywords in template.keywords:
        for q_pattern in template.question_patterns:
            for c_pattern in template.context_patterns:
                if sample_idx >= num_samples:
                    break
                
                # Use first keyword as concept placeholder
                concept = keywords[0]
                
                # Generate question and context
                try:
                    question = q_pattern.format(concept=concept, action=concept, entity=concept)
                    context = c_pattern.format(concept=concept, action=concept, entity=concept)
                except KeyError:
                    question = q_pattern.replace("{concept}", concept).replace("{action}", concept).replace("{entity}", concept)
                    context = c_pattern.replace("{concept}", concept).replace("{action}", concept).replace("{entity}", concept)
                
                # Shuffle keywords and take 3-5
                shuffled_kw = keywords.copy()
                random.shuffle(shuffled_kw)
                num_tokens = random.randint(3, min(5, len(shuffled_kw)))
                tokens = shuffled_kw[:num_tokens]
                
                # Generate probabilities
                probs = generate_probs(len(tokens))
                
                # Select 1-2 target files
                num_targets = random.randint(1, min(2, len(template.target_files)))
                targets = random.sample(template.target_files, num_targets)
                
                sample = TrainingSample(
                    sample_id=f"{template.domain}_{sample_idx:03d}",
                    confused_tokens=tokens,
                    confused_probs=probs,
                    original_query=question,
                    generated_context=context,
                    relevant_files=targets,
                    domain=template.domain,
                    difficulty=template.difficulty,
                )
                samples.append(sample)
                sample_idx += 1
    
    return samples[:num_samples]


# Generate samples from all templates
all_samples = []
samples_per_template = 15  # Generate 15 samples per template

for template in QUESTION_TEMPLATES:
    template_samples = generate_samples_from_template(template, samples_per_template)
    all_samples.extend(template_samples)

# Shuffle
random.shuffle(all_samples)

# Re-index
for i, sample in enumerate(all_samples):
    sample.sample_id = f"{sample.domain}_{i:03d}"

print(f"Generated {len(all_samples)} total samples")
print(f"\nSamples by domain:")
domain_counts = {}
for s in all_samples:
    domain_counts[s.domain] = domain_counts.get(s.domain, 0) + 1
for domain, count in sorted(domain_counts.items()):
    print(f"  {domain}: {count}")

print(f"\nSamples by difficulty:")
diff_counts = {}
for s in all_samples:
    diff_counts[s.difficulty] = diff_counts.get(s.difficulty, 0) + 1
for diff, count in sorted(diff_counts.items()):
    print(f"  {diff}: {count}")

In [ ]:
# Show sample examples
print("Sample examples:\n")
for i in range(3):
    s = all_samples[i]
    print(f"[{s.sample_id}] ({s.domain}, {s.difficulty})")
    print(f"  Query: {s.original_query}")
    print(f"  Context: {s.generated_context}")
    print(f"  Tokens: {s.confused_tokens}")
    print(f"  Probs: {s.confused_probs}")
    print(f"  Files: {s.relevant_files}")
    print()

## 4. Create Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Stratify by domain for balanced split
domains = [s.domain for s in all_samples]

train_samples, test_samples = train_test_split(
    all_samples,
    test_size=0.2,  # 80/20 split
    random_state=SEED,
    stratify=domains
)

print(f"Training samples: {len(train_samples)}")
print(f"Test samples: {len(test_samples)}")

# Verify distribution
print("\nTraining set distribution:")
train_domains = {}
for s in train_samples:
    train_domains[s.domain] = train_domains.get(s.domain, 0) + 1
for d, c in sorted(train_domains.items()):
    print(f"  {d}: {c}")

print("\nTest set distribution:")
test_domains = {}
for s in test_samples:
    test_domains[s.domain] = test_domains.get(s.domain, 0) + 1
for d, c in sorted(test_domains.items()):
    print(f"  {d}: {c}")

## 5. Save Data Files

In [ ]:
# Convert to JSON-serializable format
def sample_to_dict(sample: TrainingSample) -> dict:
    return {
        "sample_id": sample.sample_id,
        "confused_tokens": sample.confused_tokens,
        "confused_probs": sample.confused_probs,
        "original_query": sample.original_query,
        "generated_context": sample.generated_context,
        "relevant_files": sample.relevant_files,
        "domain": sample.domain,
        "difficulty": sample.difficulty,
    }

# Save training data
train_data = [sample_to_dict(s) for s in train_samples]
with open('training_samples_large.json', 'w') as f:
    json.dump(train_data, f, indent=2)
print(f"Saved {len(train_data)} training samples to training_samples_large.json")

# Save test data
test_data = [sample_to_dict(s) for s in test_samples]
with open('test_samples.json', 'w') as f:
    json.dump(test_data, f, indent=2)
print(f"Saved {len(test_data)} test samples to test_samples.json")

# Save file list
with open('file_list_large.json', 'w') as f:
    json.dump(ALL_FILES, f, indent=2)
print(f"Saved {len(ALL_FILES)} files to file_list_large.json")

# Save combined (for reference)
combined_data = {
    "train": train_data,
    "test": test_data,
    "files": ALL_FILES,
    "metadata": {
        "total_samples": len(all_samples),
        "train_size": len(train_samples),
        "test_size": len(test_samples),
        "num_files": len(ALL_FILES),
        "domains": list(domain_counts.keys()),
    }
}
with open('dataset_complete.json', 'w') as f:
    json.dump(combined_data, f, indent=2)
print("Saved complete dataset to dataset_complete.json")

In [ ]:
# Download files in Colab
try:
    from google.colab import files
    files.download('training_samples_large.json')
    files.download('test_samples.json')
    files.download('file_list_large.json')
    files.download('dataset_complete.json')
except:
    print("Files saved locally (not in Colab)")

## 6. Summary

### Generated Data

| File | Contents | Size |
|------|----------|------|
| `training_samples_large.json` | Training samples | ~120 samples |
| `test_samples.json` | Held-out test samples | ~30 samples |
| `file_list_large.json` | All file paths | ~50 files |
| `dataset_complete.json` | Combined dataset | All above |

### Next Steps

1. **Upload these files** to the Week 10 or Week 11 notebook
2. **Train** on `training_samples_large.json`
3. **Evaluate** on `test_samples.json`

With 120+ training samples (vs 21 before), the learned model should:
- Learn generalizable patterns
- Outperform heuristic on test set
- Not overfit to training data